# 08 · Agentic search, and evaluating the trace

**Deck section 8** · slides 82–86

Decompose a complex question into retrieval steps, choose a tool per step, iterate until the
evidence is sufficient or a stop condition fires. The sufficiency check is the whole design:
without it the loop either stops too early — a two-hop question answered from one hop, with
full confidence — or never stops, and one hard question costs forty times a normal one.

**By the end you can**

- run the loop and read its trace turn by turn
- name every stop condition as a config value with a default, before writing the loop
- justify single-shot versus agentic with a cost multiplier you measured
- score the *trace* — including evidence retention, the metric almost nobody instruments


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import time
import numpy as np
import pandas as pd
import raglab
from raglab import agent, viz, tables, catalog, costs, metrics, pipeline
viz.reset_figures("8."); tables.reset_tables("8.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)
multi = [q for q in bundle.questions
         if q.hops >= 2 and q.question_type != "null" and q.persona == "analyst"][:40]
single = [q for q in bundle.questions
          if q.hops == 1 and q.question_type != "null" and q.persona == "analyst"][:40]
print(f"{len(multi)} multi-hop and {len(single)} single-hop questions for this section")

---

## 8.1 The loop

Five steps, and one of them is load-bearing.


In [ ]:
viz.hld([
    dict(name="1 · Decompose", tone="index", nodes=[
        ("Break into sub-questions", "with an explicit dependency order — “who acquired B, and "
         "what was A's revenue that year” is two hops, not one query")]),
    dict(name="2 · Select a tool", tone="query", nodes=[
        ("Lexical", "identifiers"), ("Dense", "concepts"), ("Grep", "repositories"),
        ("SQL", "aggregates"), ("Live API", "anything the index cannot hold")]),
    dict(name="3 · Retrieve and read", tone="index", nodes=[
        ("Append to working evidence", "with provenance. Never overwrite — the trace is the "
         "audit record")]),
    dict(name="4 · Sufficiency check", tone="warn", nodes=[
        ("Does the evidence answer every sub-question?",
         "a separate, cheap model call with a strict schema — not a vibe inside the main "
         "prompt. NO loops with a refined sub-question; YES falls through")]),
    dict(name="5 · Synthesise and cite", tone="control", nodes=[
        ("Answer from working evidence only", "source IDs, and an explicit abstention path")]),
], title="The agentic search loop", kicker="HLD",
   caption="Step 4 is the design. Everything else is plumbing that a single-shot system "
           "already has.", source="Deck slide 83")

In [ ]:
q = next(x for x in multi if "chief executive" in x.query or "runs the business" in x.query)
print(f"QUESTION\n  {q.query}\n")

subs = agent.decompose(q.query)
print("1 · DECOMPOSE")
for i, s in enumerate(subs, 1):
    print(f"     {i}. {s}")

print("\n2 · SELECT A TOOL (per sub-question)")
for s in subs:
    print(f"     {agent.choose_tool(s):<8}  ← {s[:64]}")

print("\n   Tool choice comes from the shape of the sub-question: an identifier goes lexical")
print("   because an embedding will find related incidents and slide past ERR_CONN_RESET;")
print("   conceptual phrasing goes dense; everything else goes hybrid, which is the honest")
print("   default rather than a decision.")

In [ ]:
# 3 and 4, on real evidence.
hits = pipe.retriever.search(q.query, pipe.cfg)
ranked = pipe.reranker.rerank(q.query, hits, depth=pipe.cfg.rerank_depth)[:8]
check = agent.sufficiency_check(subs, ranked)

print("3 · RETRIEVE AND READ — first turn only")
for h in ranked[:4]:
    print(f"     {h.score:.3f}  {h.doc_id:<10} {h.text[:58]}")

print(f"\n4 · SUFFICIENCY CHECK")
print(f"     covered {check['covered']}/{check['total']} sub-questions")
print(f"     sufficient: {check['sufficient']}")
for m in check["missing"]:
    print(f"     still missing: {m[:66]}")

---

## 8.2 Running the loop

Watch it turn by turn. Note the re-anchoring: the original question text is carried into every
query after the first, which is the cheapest defence against the drift the deck warns about.


In [ ]:
budget = agent.AgentBudget(max_turns=5, max_evidence_tokens=6000, wall_clock_s=20)
loop = agent.AgenticSearch(pipe, budget)

t0 = time.perf_counter()
result = loop.run(q.query, verbose=True)
elapsed = (time.perf_counter() - t0) * 1000

print(f"\nstop reason   {result.stop_reason}")
print(f"turns         {result.turn_count}")
print(f"evidence      {len(result.working_evidence)} chunks gathered, "
      f"{len(result.final_context_ids)} packed")
print(f"wall clock    {elapsed:.0f} ms")
print(f"\nanswer        {result.answer[:260]}")
print(f"gold          {q.answer}")

gm, _ = metrics.resolve_gold(q, pipe.chunks)
print(f"\nfull-chain recall of the final context: "
      f"{metrics.full_chain_recall(result.final_context_ids, gm):.0f}")
print(f"answer correct: {metrics.answer_correct(result.answer, q.answer):.0f}")

**Read that output before moving on — the loop stopped on `sufficiency satisfied` and the
answer is wrong.**

That is not a broken demo, it is the deck's fourth agentic failure mode reproduced on the
first try: *premature confidence*. The sufficiency check counted every sub-question as
covered because chunks mentioning the entity and chunks mentioning "chief executive" were both
present — and then the extractive reader quoted the wrong one. Two lessons sit inside that:

1. **A coverage-based sufficiency check measures topical presence, not entailment.** It cannot
   tell "a chunk about Tessera" from "the chunk naming the acquirer's CEO". A real sufficiency
   call — a cheap model with a strict schema, which is what the deck specifies — reads the
   evidence and answers *can this question now be answered*, which is a different question.
2. **The stop-decision is a metric with its own precision and recall**, scored against human
   judgment. §8.6 measures it. A loop that stops confidently on partial evidence is more
   expensive than one that runs an extra turn, because it produces a wrong answer instead of
   a slow one.

Keeping this in the notebook rather than picking a question that works is deliberate. You
will meet this failure, and it looks exactly like success from the outside.


In [ ]:
turns = pd.DataFrame([{
    "turn": t.n, "tool": t.tool,
    "sub-question": t.sub_question[:46],
    "new chunks": len(t.new_chunk_ids),
    "total evidence": len(t.all_chunk_ids),
    "sufficient?": "YES" if t.sufficient else f"no ({len(t.missing)} missing)",
    "ms": round(t.elapsed_ms),
} for t in result.turns])
tables.show(turns, title="The trace, turn by turn", kicker="One question",
            caption="Every row is an audit record: what was asked, which tool answered, what "
                    "arrived that was new, and what the sufficiency check said about it.",
            emphasize="sufficient?")

---

## 8.3 Stop conditions and budget guards

Every one of these is a config value with a default, written down **before** the loop was
written. Four say you succeeded; four say you ran out.


In [ ]:
tables.show(pd.DataFrame([
    ["Sufficiency satisfied", "success", "every sub-question has supporting evidence with a "
     "source ID", f"threshold = {loop.sufficiency_threshold}", "the only clean exit"],
    ["No new information", "success", "the last turn returned only chunks already held",
     "automatic", "searching further cannot help"],
    ["Confidence plateau", "success", "two consecutive turns produce no change in the verdict",
     "automatic", "distinguishes 'done' from 'stuck'"],
    ["Turn cap", "exhausted", "hard maximum on retrieval turns",
     f"max_turns = {budget.max_turns}", "typically 4–8"],
    ["Token budget", "exhausted", "a cumulative cap across the whole trace, not per call",
     f"max_evidence_tokens = {budget.max_evidence_tokens:,}",
     "per-call caps do not bound a loop"],
    ["Wall-clock deadline", "exhausted", "return the best partial answer rather than blowing "
     "the SLA", f"wall_clock_s = {budget.wall_clock_s}", "the user is still waiting"],
    ["Repeat detector", "exhausted", "the same normalised query issued twice ends the loop",
     "automatic", "catches tool thrash"],
], columns=["Condition", "Kind", "What it means", "Config value here", "Note"]),
    title="Stop conditions, as configuration",
    kicker="Budget guards", source="Deck slide 85", emphasize="Condition",
    highlight_rows=lambda r: r["Kind"] == "exhausted")

tables.callout(
    "A budget exhaustion must produce an <b>explicit partial answer with a stated gap</b> — "
    "“I found A but could not confirm B” — never a confident synthesis of half the evidence. "
    "The stop reason belongs in the trace and, usually, in the UI. "
    "<code>AgenticSearch.run()</code> appends a PARTIAL line whenever it stops for any reason "
    "other than sufficiency; that is not decoration, it is the difference between a slow "
    "answer and a wrong one.", kind="warn")

In [ ]:
# Demonstrate each exhaustion condition firing.
demos = []
hard = [x for x in multi if x.hops >= 2][:6]

tight = agent.AgenticSearch(pipe, agent.AgentBudget(max_turns=1))
r1 = tight.run(hard[0].query)
demos.append(["Turn cap = 1", r1.stop_reason, r1.turn_count,
              "PARTIAL" in r1.answer, r1.answer[-70:].strip()])

deadline = agent.AgenticSearch(pipe, agent.AgentBudget(max_turns=8, wall_clock_s=0.001))
r2 = deadline.run(hard[1].query)
demos.append(["Wall clock = 1 ms", r2.stop_reason, r2.turn_count,
              "PARTIAL" in r2.answer, r2.answer[-70:].strip()])

strict = agent.AgenticSearch(pipe, agent.AgentBudget(max_turns=6),
                             sufficiency_threshold=0.95)
r3 = strict.run(hard[2].query)
demos.append(["Sufficiency threshold 0.95 (unreachable)", r3.stop_reason, r3.turn_count,
              "PARTIAL" in r3.answer, r3.answer[-70:].strip()])

normal = agent.AgenticSearch(pipe, agent.AgentBudget(max_turns=6))
r4 = normal.run(hard[3].query)
demos.append(["Default budget", r4.stop_reason, r4.turn_count,
              "PARTIAL" in r4.answer, r4.answer[-70:].strip()])

tables.show(pd.DataFrame(demos, columns=[
    "Configuration", "Stop reason recorded", "Turns", "States the gap?", "Tail of the answer"]),
    title="Each guard, fired on purpose",
    kicker="Demonstrated", emphasize="Stop reason recorded",
    caption="The 'states the gap' column is the one that matters in a client system. An agent "
            "that runs out of budget and says so is operating correctly; one that runs out "
            "and synthesises anyway is an incident waiting for a date.")

---

## 8.4 Does the loop earn its cost?

Default to single-shot. Agentic search is a cost and latency multiplier you should have to
justify — so measure the multiplier rather than quoting the deck's "3–20×".


In [ ]:
catalog.AGENTIC_VS_SINGLE.show()

In [ ]:
def measure(questions, mode):
    '''mode: single | agentic | escalate'''
    recs = []
    for x in questions:
        gm, _ = metrics.resolve_gold(x, pipe.chunks)
        acl = bundle.personas.get(x.persona)
        t0 = time.perf_counter()
        if mode == "single":
            tr = pipe.run(x.query, qid=x.qid, acl_groups=acl)
            turns, ids, answer = 1, tr.packed_ids, tr.answer
            tokens = tr.usage.get("input_tokens", 0) + tr.usage.get("output_tokens", 0)
        else:
            first = pipe.run(x.query, qid=x.qid, acl_groups=acl)
            if mode == "escalate":
                sub = agent.decompose(x.query)
                ok = agent.sufficiency_check(sub, first._selected)["sufficient"]
                if ok:
                    turns, ids, answer = 1, first.packed_ids, first.answer
                    tokens = (first.usage.get("input_tokens", 0)
                              + first.usage.get("output_tokens", 0))
                    recs.append(dict(qid=x.qid, turns=turns, ms=(time.perf_counter()-t0)*1000,
                                     tokens=tokens,
                                     full_chain_recall=metrics.full_chain_recall(ids, gm),
                                     answer_correct=metrics.answer_correct(answer, x.answer),
                                     escalated=0))
                    continue
            res = loop.run(x.query, acl_groups=acl)
            turns, ids, answer = res.turn_count, res.final_context_ids, res.answer
            tokens = res.usage.get("input_tokens", 0) + res.usage.get("output_tokens", 0)
            tokens *= max(1, turns)      # each turn pays its own retrieval + sufficiency call
        recs.append(dict(qid=x.qid, turns=turns, ms=(time.perf_counter() - t0) * 1000,
                         tokens=tokens,
                         full_chain_recall=metrics.full_chain_recall(ids, gm),
                         answer_correct=metrics.answer_correct(answer, x.answer),
                         escalated=1 if mode != "single" and turns > 1 else 0))
    return recs


rates = costs.Rates()
comparison = {}
for mode in ("single", "agentic", "escalate"):
    comparison[mode] = measure(multi, mode)

rows = []
for mode, recs in comparison.items():
    tok = np.mean([r["tokens"] for r in recs])
    rows.append([
        {"single": "single-shot", "agentic": "agentic (always loop)",
         "escalate": "escalate on sufficiency failure"}[mode],
        round(np.mean([r["turns"] for r in recs]), 2),
        round(np.mean([r["ms"] for r in recs])),
        int(tok),
        f"${rates.cost(input_tokens=tok*0.9, output_tokens=tok*0.1):.5f}",
        round(np.mean([r["full_chain_recall"] for r in recs]), 3),
        round(np.mean([r["answer_correct"] for r in recs]), 3),
        f"{np.mean([r['escalated'] for r in recs]):.0%}",
    ])
frame = pd.DataFrame(rows, columns=[
    "Mode", "Avg turns", "Avg ms", "Avg tokens", "Cost/query", "Full-chain recall",
    "Answer correct", "Share escalated"])
tables.show(frame, title=f"Single-shot, agentic and escalation on {len(multi)} multi-hop "
                         "questions",
            kicker="Measured", emphasize="Mode",
            caption="The middle row is what people mean by 'agentic RAG'. The bottom row is "
                    "what you should usually build: most traffic pays single-shot cost, and "
                    "hard questions get the budget they need.",
            highlight_rows=lambda r: r["Mode"].startswith("escalate"))

mult = (np.mean([r["tokens"] for r in comparison["agentic"]])
        / np.mean([r["tokens"] for r in comparison["single"]]))
print(f"measured cost multiplier, always-loop vs single-shot: {mult:.1f}×")

In [ ]:
# The blended picture: run the escalation policy over a realistic traffic mix.
mixed = multi + single
mixed_recs = measure(mixed, "escalate")
single_recs = measure(mixed, "single")

hard_share = np.mean([r["escalated"] for r in mixed_recs])
tok_esc = np.mean([r["tokens"] for r in mixed_recs])
tok_single = np.mean([r["tokens"] for r in single_recs])

tables.keyvalue([
    ("Traffic mix", f"{len(multi)} multi-hop + {len(single)} single-hop"),
    ("Share the policy escalated", f"{hard_share:.0%}"),
    ("Blended tokens per query", f"{tok_single:,.0f} single-shot → {tok_esc:,.0f} escalating "
                                 f"({tok_esc/tok_single:.2f}×)"),
    ("Blended full-chain recall", f"{np.mean([r['full_chain_recall'] for r in single_recs]):.3f}"
                                  f" → {np.mean([r['full_chain_recall'] for r in mixed_recs]):.3f}"),
    ("Blended answer correctness", f"{np.mean([r['answer_correct'] for r in single_recs]):.3f}"
                                   f" → {np.mean([r['answer_correct'] for r in mixed_recs]):.3f}"),
], title="Escalation, priced against a realistic traffic mix",
   kicker="The middle path",
   caption="This is the answer to interview Q5's cost problem: you do not make hard questions "
           "cheaper, you stop paying hard-question cost on easy ones.")

---

## 8.5 What goes wrong

Five failure modes, and each one has a defence that is already in the loop.


In [ ]:
drift_rows = []

# Query drift -- measured as vocabulary overlap between the original question and turn N.
from raglab.embed import tokenize
overlaps = []
for x in multi[:15]:
    res = loop.run(x.query, acl_groups=bundle.personas.get(x.persona))
    q0 = set(tokenize(x.query))
    for t in res.turns:
        ov = len(q0 & set(tokenize(t.query_issued))) / max(1, len(q0))
        overlaps.append((t.n, ov))
by_turn = {}
for n, ov in overlaps:
    by_turn.setdefault(n, []).append(ov)
drift_rows.append(["Query drift",
                   "By turn four the agent is searching for something adjacent to the "
                   "original question",
                   " · ".join(f"turn {n}: {np.mean(v):.2f}" for n, v in sorted(by_turn.items())),
                   "Re-anchor on the original question text every turn — the loop concatenates "
                   "it into every query after the first"])

# Evidence bloat.
sizes = []
for x in multi[:15]:
    res = loop.run(x.query, acl_groups=bundle.personas.get(x.persona))
    sizes.append((len(res.working_evidence), len(res.final_context_ids)))
drift_rows.append(["Evidence bloat",
                   "Working evidence grows past the context budget and the earliest, often "
                   "best, results get dropped",
                   f"gathered {np.mean([a for a, _ in sizes]):.1f} chunks, packed "
                   f"{np.mean([b for _, b in sizes]):.1f}",
                   "A cumulative token cap across the trace, and a compacted summary carried "
                   "between turns instead of full text"])

# Tool thrash.
tools_used = []
for x in multi[:15]:
    res = loop.run(x.query, acl_groups=bundle.personas.get(x.persona))
    tools_used.append(len({t.tool for t in res.turns}) / max(1, res.turn_count))
drift_rows.append(["Tool thrash",
                   "The same query re-issued to three tools because none returned a confident "
                   "result",
                   f"distinct tools per turn: {np.mean(tools_used):.2f}",
                   "Deduplicate issued queries — the repeat detector ends the loop on a "
                   "normalised repeat"])

# Premature confidence.
early = [r for r in comparison["agentic"] if r["turns"] == 1 and r["full_chain_recall"] == 0]
drift_rows.append(["Premature confidence",
                   "The sufficiency check passes on partial evidence, so a two-hop question "
                   "gets a one-hop answer with full confidence",
                   f"{len(early)} of {len(comparison['agentic'])} multi-hop questions stopped "
                   "after one turn without the full chain",
                   "This is the metric to tune the threshold against — precision and recall of "
                   "the stop decision against human judgment"])

# Cost blowout.
tok = [r["tokens"] for r in comparison["agentic"]]
drift_rows.append(["Cost blowout",
                   "Nothing bounds the loop and a single hard question costs forty times a "
                   "normal one",
                   f"p50 {np.percentile(tok, 50):,.0f} tokens, p95 {np.percentile(tok, 95):,.0f} "
                   f"({np.percentile(tok, 95)/np.percentile(tok, 50):.1f}× the median)",
                   "A cumulative token cap and a turn cap — and watch the tail, not the mean"])

tables.show(pd.DataFrame(drift_rows, columns=[
    "Failure mode", "What it looks like", "Measured here", "The defence in this loop"]),
    title="What goes wrong in an agentic loop",
    kicker="Failure modes", source="Deck slide 83", emphasize="Failure mode")

---

## 8.6 Evaluating the trace, not just the answer

Answer-only scoring cannot tell a lucky agent from a good one. Six trace properties, and the
fifth is the one almost nobody instruments.


In [ ]:
catalog.TRACE_EVAL.show()

In [ ]:
scores = []
for x in multi[:25]:
    gm, _ = metrics.resolve_gold(x, pipe.chunks)
    if not gm:
        continue
    res = loop.run(x.query, acl_groups=bundle.personas.get(x.persona))
    s = agent.score_trace(res, gm, min_turns=max(1, len(gm) // 2))
    s["qid"] = x.qid
    scores.append(s)

agg = pd.DataFrame(scores)
summary = pd.DataFrame([{
    "Decomposition quality": round(agg["decomposition_quality"].mean(), 3),
    "Turn efficiency": round(agg["turn_efficiency"].mean(), 3),
    "Cumulative evidence recall": round(agg["cumulative_evidence_recall"].mean(), 3),
    "Evidence retention": round(agg["evidence_retention"].mean(), 3),
    "Found then lost (total)": int(agg["found_then_lost"].sum()),
    "Stopped on sufficiency": f"{agg['stopped_well'].mean():.0%}",
}])
tables.show(summary.T.reset_index().rename(columns={"index": "Trace property", 0: "Value"}),
            title=f"Trace scores over {len(scores)} multi-hop questions",
            kicker="Scored the trace", emphasize="Trace property",
            caption="Read cumulative evidence recall against evidence retention. The gap "
                    "between them is gold the loop found and then discarded while packing.")

gap = agg["cumulative_evidence_recall"].mean() - agg["evidence_retention"].mean()
print(f"cumulative evidence recall  {agg['cumulative_evidence_recall'].mean():.3f}")
print(f"evidence retention          {agg['evidence_retention'].mean():.3f}")
print(f"gap (found, then thrown away) {gap:+.3f}   "
      f"— {int(agg['found_then_lost'].sum())} gold items across {len(scores)} questions\n")

if gap < 0.01:
    print("No retention gap on this run, and the reason is structural rather than lucky:")
    print(f"the loop gathers ~{np.mean([len(r['_'] if False else []) for r in []] or [0]):.0f}"
          if False else
          "the loop gathers only a little more evidence than k, so almost nothing has to")
    print("compete for a slot. The gap opens when the loop runs long enough that working")
    print("evidence exceeds the packing budget — raise max_turns, lower k, and it appears.")
    print("Instrument it now anyway: it is invisible until the day it is not, and by then")
    print("you are debugging a multi-turn system without the one number that explains it.")

In [ ]:
viz.bars(["cumulative evidence recall\n(found anywhere in the trace)",
          "evidence retention\n(survived into the final context)"],
         {"score": [agg["cumulative_evidence_recall"].mean(), agg["evidence_retention"].mean()]},
         title="Evidence retention: the metric that decides whether the loop was worth it",
         kicker="Trace evaluation", ylabel="score",
         caption="Every point of that gap is evidence the agent paid to retrieve and then "
                 "discarded during packing. It is where multi-turn systems quietly lose to "
                 "single-shot ones, and it is invisible to answer-only scoring.")

In [ ]:
tables.callout(
    "If the gap above is large, the fix is <b>not</b> more turns — more turns make it worse by "
    "adding candidates that compete for the same k slots. The fix is in packing: carry a "
    "compacted summary of earlier evidence between turns instead of the full text, and give "
    "gold found in early turns a reserved slot rather than making it re-win the ranking "
    "against everything found since."
    "<br><br>That is the same insight as notebook 02's per-entity packing quota, in a "
    "different costume: a global relevance ranking cannot express a constraint about "
    "<i>coverage</i>, so the constraint belongs in the packer.", kind="note",
    title="What to do about a retention gap")

---

## 8.7 The interview


In [ ]:
tables.show(pd.DataFrame([
    ["When would you use agentic search over single-shot RAG?",
     "Whether you default to the expensive option",
     "When later hops depend on what earlier hops returned. Otherwise single-shot — and the "
     f"honest middle is escalation: measured here, {hard_share:.0%} of a realistic mix "
     f"escalated and the blended cost was {tok_esc/tok_single:.2f}× single-shot."],
    ["How do you stop an agent loop?",
     "Whether stop conditions were designed or discovered",
     "Four success conditions and four exhaustion conditions, every one a config value with a "
     "default written down before the loop. And a budget exhaustion produces an explicit "
     "partial answer with a stated gap."],
    ["How do you evaluate an agent, not just its answer?",
     "Whether you have ever debugged a multi-turn system",
     "Score the trace: decomposition coverage, tool-selection accuracy, turn efficiency, "
     "cumulative evidence recall, evidence retention, and stop-decision precision/recall. "
     "Lead with evidence retention — most people have never instrumented it."],
    ["Your agent costs $0.90 on hard questions. Finance wants $0.15.",
     "Whether you optimise the distribution or the worst case",
     "Get the share of hard traffic first; escalate rather than looping by default; cache the "
     "prefix; compact evidence between turns; small model for decomposition and sufficiency. "
     "Refuse to drop the grounding checks and the trace, and quantify the residual."],
], columns=["Question", "What the panel is testing", "What a strong answer covers"]),
    title="Interview questions: agentic search",
    kicker="Section 8 · interview", emphasize="Question")

---

## 8.8 Checkpoint

1. Your agent's answer quality is flat but its cost tripled. Which trace metric do you look at
   first?
2. The sufficiency check passes on 95% of first turns. Is that good?
3. Name the one thing you would refuse to remove from an agent loop under cost pressure.


In [ ]:
print("1 ·  Turn efficiency, then cumulative evidence recall. If turns went up and cumulative")
print("     recall did not, the extra turns found nothing — that is tool thrash or query")
print("     drift, and the repeat detector plus re-anchoring are the fixes. If cumulative")
print("     recall DID go up while retention did not, you are paying to find evidence and")
print(f"     then discarding it: measured here, a gap of {gap:+.3f}.\n")

print("2 ·  Not without knowing the base rate. If 95% of your traffic is genuinely single-hop,")
print("     it is exactly right. If a third is multi-hop, the check is rubber-stamping partial")
print("     evidence — premature confidence, and the most expensive failure in the list")
print("     because it produces a confident wrong answer rather than a slow one.")
print(f"     Measured here on multi-hop questions: {len(early)} of "
      f"{len(comparison['agentic'])} stopped after one turn without the full chain.\n")

print("3 ·  The grounding and abstention checks, and the trace. Both are cheap and both are")
print("     what stop a wrong answer from becoming an incident. Everything else on the loop")
print("     is negotiable with a number attached — those two you name as the line.")

---

## What carries forward

- The sufficiency check is the design. Make it a separate, cheap, schema-constrained call.
- Every stop condition is a config value with a default, decided before the loop is written —
  and an exhausted budget produces a stated gap, never a confident synthesis.
- Escalate rather than loop by default. Most traffic pays single-shot cost; hard questions get
  the budget.
- Score the trace. Evidence retention is where multi-turn systems quietly lose, and the fix
  lives in the packer rather than in more turns.

**Next:** `09_capstone_build.ipynb` — the deck's build brief, run end to end: harness first,
baseline, three measured improvements, the frozen-slice check, and a one-page decision record
scored against the rubric.
